In [0]:
from pyspark.sql import functions as F

raw_path = "s3://multi-city-weather-data-akil-2025/raw/"

df = (
    spark.read
    .option("multiLine", True)
    .json(raw_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

display(
    df.select(
        "_source_file",
        "latitude",
        "longitude"
    )
)

In [0]:
df.printSchema()

In [0]:
display(df.select("hourly"))

In [0]:
from pyspark.sql import functions as F

df = df.withColumn(
    "city",
    F.when((F.col("latitude") == 13.0827) & (F.col("longitude") == 80.2707), "Chennai")
     .when((F.col("latitude") == 12.9716) & (F.col("longitude") == 77.5946), "Bengaluru")
     .when((F.col("latitude") == 17.3850) & (F.col("longitude") == 78.4867), "Hyderabad")
     .when((F.col("latitude") == 19.0760) & (F.col("longitude") == 72.8777), "Mumbai")
     .when((F.col("latitude") == 28.6139) & (F.col("longitude") == 77.2090), "Delhi")
     .when((F.col("latitude") == 11.0168) & (F.col("longitude") == 76.9558), "Coimbatore")
     .otherwise("Unknown")
)

display(
    df.select("city", "latitude", "longitude")
)

In [0]:
df = df.withColumn(
    "city",
    F.initcap(
        F.regexp_extract(
            F.element_at(F.split(F.col("_source_file"), "/"), -1),
            r"^([^.]+)",
            1
        )
    )
)

display(
    df.select(
        "city",
        "latitude",
        "longitude",
        "_source_file"
    )
)

In [0]:
hourly_df = (
    df
    .select(
        "city",
        "latitude",
        "longitude",
        F.posexplode("hourly.time").alias("hour_index", "timestamp"),
        "hourly"
    )
)

display(hourly_df)

In [0]:
bronze_df = (
    hourly_df
    .select(
        "city",
        "latitude",
        "longitude",
        "timestamp",
        F.col("hourly.temperature_2m")[F.col("hour_index")].alias("temperature_2m"),
        F.col("hourly.relative_humidity_2m")[F.col("hour_index")].alias("relative_humidity_2m"),
        F.col("hourly.precipitation")[F.col("hour_index")].alias("precipitation"),
        F.col("hourly.wind_speed_10m")[F.col("hour_index")].alias("wind_speed_10m")
    )
)

display(bronze_df)


In [0]:
from pyspark.sql import functions as F

null_check = bronze_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in bronze_df.columns
    ]
)

display(null_check)

In [0]:
duplicate_count = (
    bronze_df
    .groupBy("city", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_count)

In [0]:
quality_check = bronze_df.select(
    F.sum(
        F.when(
            (F.col("relative_humidity_2m") < 0) |
            (F.col("relative_humidity_2m") > 100),
            1
        ).otherwise(0)
    ).alias("invalid_humidity"),

    F.sum(
        F.when(
            F.col("precipitation") < 0,
            1
        ).otherwise(0)
    ).alias("invalid_precipitation"),

    F.sum(
        F.when(
            F.col("wind_speed_10m") < 0,
            1
        ).otherwise(0)
    ).alias("invalid_wind_speed"),

    F.sum(
        F.when(
            (F.col("temperature_2m") < -90) |
            (F.col("temperature_2m") > 60),
            1
        ).otherwise(0)
    ).alias("invalid_temperature")
)

display(quality_check)

In [0]:
bronze_table = "weather_bronze"

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

In [0]:
spark.sql("""
    SELECT *
    FROM weather_bronze
    LIMIT 20
""").display()

silver_layer


In [0]:
silver_df = spark.table("weather_bronze")

display(silver_df)


In [0]:
silver_df.printSchema()

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "timestamp",
    F.to_timestamp("timestamp")
)

In [0]:
silver_df = (
    silver_df
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month", F.month("timestamp"))
)

In [0]:
silver_df = spark.table("weather_bronze")
silver_df.printSchema()

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("weather_bronze")

silver_df = (
    silver_df
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month", F.month("timestamp"))
)

silver_df.printSchema()

In [0]:
display(
    silver_df.select(
        "city",
        "timestamp",
        "date",
        "hour",
        "day_of_week",
        "month",
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    )
)

In [0]:
silver_df = (
    silver_df
    .dropDuplicates(["city", "timestamp"])
)

display(silver_df)

In [0]:
print("Silver row count:", silver_df.count())

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_silver")

In [0]:
spark.sql("""
    SELECT *
    FROM weather_silver
    LIMIT 20
""").display()

In [0]:
print("Silver row count:", spark.table("weather_silver").count())

spark.table("weather_silver").printSchema()

gold_layer


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_location = (
    spark.table("weather_silver")
    .select(
        "city",
        "latitude",
        "longitude"
    )
    .dropDuplicates(["city"])
    .withColumn(
        "location_id",
        F.row_number().over(
            Window.orderBy("city")
        )
    )
    .select(
        "location_id",
        "city",
        "latitude",
        "longitude"
    )
)

display(dim_location)

In [0]:
dim_location.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_location")

In [0]:
spark.sql("""
    SELECT *
    FROM dim_location
    ORDER BY location_id
""").display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_date = (
    spark.table("weather_silver")
    .select("date")
    .dropDuplicates()
    .withColumn(
        "date_id",
        F.row_number().over(
            Window.orderBy("date")
        )
    )
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .select(
        "date_id",
        "date",
        "year",
        "month",
        "month_name",
        "day",
        "day_of_week"
    )
)

display(dim_date)

In [0]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

In [0]:
spark.sql("""
    SELECT *
    FROM dim_date
    ORDER BY date_id
""").display()

In [0]:
from pyspark.sql import functions as F

fact_weather = (
    spark.table("weather_silver")
    .join(
        spark.table("dim_location"),
        on="city",
        how="left"
    )
    .join(
        spark.table("dim_date"),
        on="date",
        how="left"
    )
    .select(
        F.monotonically_increasing_id().alias("weather_id"),
        "location_id",
        "date_id",
        "timestamp",
        F.col("temperature_2m").alias("temperature"),
        F.col("relative_humidity_2m").alias("humidity"),
        "precipitation",
        F.col("wind_speed_10m").alias("wind_speed")
    )
)

display(fact_weather)

In [0]:
fact_weather.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_weather")
    

In [0]:
print("Fact weather row count:", spark.table("fact_weather").count())

In [0]:
spark.sql("""
SELECT *
FROM fact_weather
LIMIT 20
""").display()

In [0]:
print("Locations:", spark.table("dim_location").count())
print("Dates:", spark.table("dim_date").count())
print("Weather facts:", spark.table("fact_weather").count())

In [0]:
spark.sql("""
SELECT
    f.weather_id,
    l.city,
    d.date,
    f.timestamp,
    f.temperature,
    f.humidity,
    f.precipitation,
    f.wind_speed
FROM fact_weather f
JOIN dim_location l
    ON f.location_id = l.location_id
JOIN dim_date d
    ON f.date_id = d.date_id
ORDER BY d.date, l.city, f.timestamp
LIMIT 20
""").display()